In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
import os
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

## Rename files

In [ ]:
id_map = pd.read_parquet('/s/project/uk_biobank/clean/gagnuer_stegle_fam_mapping_with_missing_dummies.parquet').rename(columns={'gagneur_eid':'individual'})
id_map

In [ ]:
a = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_results/paper_funcrvp/v1NEWsplit_deepRVAT_testsplit0.25_pops_predictions_extended.pq')
a['individual'] = a['individual'].astype(int)
c = a.merge(id_map, on='individual').drop(columns=['individual']).rename(columns={'stegle_eid':'individual'})
c

In [ ]:
emb_list = ['noemb']

for emb in emb_list:
    a = pd.read_parquet(f'/s/project/geno2pheno/funcrvp/paper_results/paper_funcrvp/v1NEWsplit_noemb_deepRVAT_testsplit0.25_{emb}_predictions_extended.pq').rename(
        columns={'common_residual': 'common_variant_residual', 'best_r2_pred': 'best_prediction', 'version': 'experiment_name'}).drop(
            columns=['phenocode', 'best_loss_pred'])
    a['individual'] = a['individual'].astype(int)
    b = a.merge(id_map, on='individual').drop(columns=['individual']).rename(columns={'stegle_eid':'individual'})
    b.to_parquet(f'/s/project/geno2pheno/funcrvp/paper_revisions/old_results/v1NEWsplit_noemb_deepRVAT_testsplit0.25_{emb}_predictions_extended.pq')

In [ ]:
base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat'

# for thresh in ['0.001', '0.005', '0.01', '0.05', '0.1', '0.5', '1.0']:
#     pred_file = f"{base_dir}/all_traits_phenopred_{thresh}.pq"
#     pd.read_parquet(pred_file).rename(columns={'pred': 'best_prediction'}).to_parquet(pred_file)

## Define filepaths and dataframes

In [ ]:
base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions'
lm_cov = f"{base_dir}/predictions/all_traits_covariates_only_phenopred_filteredv3.pq"

model_dict = {
    'rvat_0.001': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.001.pq",
    'rvat_0.005': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.005.pq",
    'rvat_0.01': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.01.pq",
    'rvat_0.05': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.05.pq",
    'rvat_0.1': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.1.pq",
    'rvat_0.5': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.5.pq",
    'rvat_1.0': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_1.0.pq",
    'rvat_0.0001nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.0001nom.pq",
    'rvat_0.001nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.001nom.pq",
    'rvat_0.01nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.01nom.pq",
    'rvat_0.05nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.05nom.pq",
    'rvat_0.1nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.1nom.pq",
    'rvat_0.5nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_0.5nom.pq",
    'rvat_1.0nom': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_1.0nom.pq",
    'funcrvp': f"{base_dir}/old_results/v1NEWsplit_deepRVAT_testsplit0.25_omics_pops_predictions_extended.pq",
    'new_funcrvp': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3/all_traits_phenopred.pq",
    # 'enformer': f"{base_dir}/old_results/v1NEWsplit_deepRVAT_testsplit0.25_enformer_small_predictions_extended.pq",
    # 'gene2vec': f"{base_dir}/old_results/v1NEWsplit_deepRVAT_testsplit0.25_gene2vec_predictions_extended.pq",
    # 'ESM2': f"{base_dir}/old_results/v1NEWsplit_deepRVAT_testsplit0.25_esm2_predictions_extended.pq",
    # 'omics': f"{base_dir}/old_results/v1NEWsplit_deepRVAT_testsplit0.25_omics_predictions_extended.pq",
    # 'pops': f"{base_dir}/old_results/v1NEWsplit_deepRVAT_testsplit0.25_pops_predictions_extended.pq",
    # 'random_emb': f"{base_dir}/old_results/v1NEWsplit_randEmb_deepRVAT_testsplit0.25_omics_pops_predictions_extended.pq",
    # 'new_omics': f"{base_dir}/funcrvp_predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/omics_d256/funcrvp_rev1_filteredv3/all_traits_phenopred.pq",
    # 'new_pops': f"{base_dir}/funcrvp_predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_d256/funcrvp_rev1_filteredv3/all_traits_phenopred.pq",
    # 'new_codon': f"{base_dir}/funcrvp_predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/codon_emb_d64/funcrvp_rev1_filteredv3/all_traits_phenopred.pq",
    # 'new_ts2_pseudobulk': f"{base_dir}/funcrvp_predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/tabula_sapiens2_pseudobulk_d512/funcrvp_rev1_filteredv3/all_traits_phenopred.pq",
    # 'new_ts2_100k': f"{base_dir}/funcrvp_predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/tabula_sapiens2_100_000_cells_d512/funcrvp_rev1_filteredv3/all_traits_phenopred.pq",
    # 'new_gene2vec': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/gene2vec_d200/funcrvp_better_filteredv3_samplingNone/all_traits_phenopred.pq",
    # 'new_ESM2': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/ESM2_PCA_d512/funcrvp_better_filteredv3_samplingNone/all_traits_phenopred.pq",
    # 'new_enformer': f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/enformer_d512/funcrvp_better_filteredv3_samplingNone/all_traits_phenopred.pq",
}

In [ ]:
df_list = []
for model_name, pred_path in tqdm(model_dict.items()):
    temp = pd.read_parquet(pred_path)
    temp['individual'] = temp['individual'].astype(str)
    temp['model_name'] = model_name
    if 'pred' in temp.columns:
        temp['best_prediction'] = temp['pred']
        temp.drop(columns=['pred'], inplace=True)
    df_list.append(temp)

dt_new = pd.concat(df_list).drop(columns=['version', 'phenocode', 'embedding', 'experiment_name'])
dt_new

In [ ]:
cov_df = pd.read_parquet(lm_cov).rename(columns={'best_prediction':'cov_pred'})
cov_df['individual'] = cov_df['individual'].astype(str)
cov_df

In [ ]:
dt_bt = dt_new.merge(cov_df[['individual', 'trait', 'cov_pred']], on=['trait', 'individual'])
dt_bt

In [ ]:
dt_bt.model_name.unique()

In [ ]:
dt_bt = dt_bt[['individual', 'model_name', 'trait', 'trait_measurement', 'cov_pred', 'best_prediction']]
dt_bt

In [ ]:
dt_wide = dt_bt.pivot(index=['individual', 'trait', 'trait_measurement', 'cov_pred'], values='best_prediction', columns='model_name').reset_index()
dt_wide

## Plot R2

In [ ]:
dt_bt

In [ ]:
# If you want R2 per trait per model:
# Define a function to calculate R2 per trait within a model group
def calculate_trait_r2(group):
    try:
        model_r2 = r2_score(group['trait_measurement'], group['best_prediction'])
    except ValueError:
        model_r2 = np.nan
    try:
        cov_r2 = r2_score(group['trait_measurement'], group['cov_pred'])
    except ValueError:
        cov_r2 = np.nan
    return pd.Series({'model_r2': model_r2, 'cov_r2': cov_r2})

# Group by both 'model_name' and 'trait'
r2_by_model_trait = dt_bt.groupby(['model_name', 'trait'], observed=True).apply(calculate_trait_r2).reset_index()

r2_by_model_trait

## Compute bootstraps

In [ ]:

# --- Modified Worker Function ---
def calculate_bootstrap_r2_multi_pred(i, data, group_col, target, pred_cols):
    """
    Calculates R2 per group for multiple prediction columns
    for a single bootstrap sample.
    """
    sample = data.sample(frac=1, replace=True, random_state=i)
    iteration_results = []

    for group_key, group in sample.groupby(group_col):
        # Calculate R2 for each prediction column
        for pred_col in pred_cols:
            try:
                # Ensure prediction column also has variance if r2_score requires it
                if group[pred_col].nunique() <= 1:
                     r2 = np.nan # R2 is undefined for constant prediction
                else:
                    r2 = r2_score(group[target], group[pred_col])
                    cov_r2 = r2_score(group[target], group['cov_pred'])

            except ValueError: # Catch potential errors during r2_score calculation
                r2 = np.nan
            except KeyError:   # Catch if a pred_col is somehow missing (shouldn't happen with sampling)
                r2 = np.nan

            iteration_results.append({
                'bootstrap_iteration': i,
                group_col: group_key,
                'model': pred_col,
                'r2': r2,
                'cov_r2': cov_r2
            })
    return iteration_results # Return list of dicts for this iteration

In [ ]:
n_bt = 10000
n_jobs = -1
prediction_cols = model_dict.keys() #['model_pred_1', 'model_pred_2', 'covariate_pred']
target_col = 'trait_measurement'
grouping_col = 'trait' # Usually 'trait', but make it explicit

# --- Parallel Execution ---
results_list_of_lists = Parallel(n_jobs=n_jobs)(
    delayed(calculate_bootstrap_r2_multi_pred)(
        i,
        dt_wide,
        grouping_col,
        target_col,
        prediction_cols
    )
    for i in tqdm(range(n_bt), desc="Bootstrapping R2 (Multi-Pred)")
)

# --- Flatten the list of lists and Create Final DataFrame ---
flat_results = [item for sublist in results_list_of_lists for item in sublist if sublist] # Ensure sublist is not empty

if flat_results:
    r2_bt_multi = pd.DataFrame(flat_results)
else:
    r2_bt_multi = pd.DataFrame(columns=['bootstrap_iteration', grouping_col, 'prediction_column', 'r2'])

# --- Display Results ---
print(f"Generated {len(r2_bt_multi)} R2 values.")

In [ ]:
r2_bt_multi.to_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap_rvat.pq', index=False)


# Compare models

In [ ]:
# r2_bt_multi = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap_rvat.pq')
r2_bt_multi = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap.pq')

r2_bt_multi['rel_delta_r2'] = (r2_bt_multi['r2'] - r2_bt_multi['cov_r2'])/r2_bt_multi['cov_r2']

n_bt = r2_bt_multi.bootstrap_iteration.nunique()
r2_bt_multi

In [ ]:
plot_df = r2_bt_multi[['bootstrap_iteration', 'trait', 'model', 'rel_delta_r2']].pivot(
    index=['bootstrap_iteration', 'trait'],
    columns='model',
    values='rel_delta_r2').reset_index()

plot_df

In [ ]:
model_list = ['funcrvp', 'omics', 'pops', 'enformer', 'gene2vec', 'ESM2', 'random_emb', 'new_codon']#, 'new_ts2_100k', 'new_ts2_pseudobulk']
# model_list = ['funcrvp', 'new_funcrvp', 'rvat_0.0001nom', 'rvat_0.001', 'rvat_0.001nom', 'rvat_0.005', 'rvat_0.01', 'rvat_0.01nom', 'rvat_0.05nom', 'rvat_0.1', 'rvat_0.1nom', 'rvat_0.05', 'rvat_0.5nom', 'rvat_0.5', 'rvat_1.0nom']


stats_plot_list =  []
for f_model in model_list:
    plot_df['r2_diff'] = plot_df[f_model] - plot_df['rvat']

    # --- Step 1: Grouped Aggregation ---
    # Group by the specified columns and aggregate r2_diff
    stats_plot = plot_df.groupby(['trait'])['r2_diff'].agg(
        # Count where r2_diff > 0. Boolean True is treated as 1, False as 0.
        N_greater=lambda x: (x > 0).sum(),
        # Count where r2_diff < 0
        N_lesser=lambda x: (x < 0).sum()
    ).reset_index() # reset_index turns the grouped indices back into columns

    # --- Step 2: Calculate p-value ---
    # Calculate the two terms for the minimum function
    term1 = (stats_plot['N_greater'] + 1) / (n_bt + 1)
    term2 = (stats_plot['N_lesser'] + 1) / (n_bt + 1)

    # Calculate the p-value using np.minimum for element-wise minimum
    stats_plot['pval'] = 2 * np.minimum(term1, term2)
    stats_plot['model'] = f_model
    stats_plot_list.append(stats_plot)

stats_df = pd.concat(stats_plot_list)
stats_df

In [ ]:
stats_df['comparison'] = stats_df.apply(
    lambda x: 'significantly better' if (x.N_lesser < 5000) & (x.pval < 0.05) 
    else ('significantly worse' if (x.N_lesser > 5000) & (x.pval < 0.05) else 'no significant difference'), 
    axis=1
)

stats_df['model'] = pd.Categorical(stats_df['model'], categories=model_list, ordered=True)

emb_map = {
    'funcrvp': 'Omics+PoPS', 
    'omics': 'Omics', 
    'pops': 'PoPS', 
    'enformer': 'Enformer', 
    'gene2vec': 'gene2vec', 
    'ESM2': 'ESM2', 
    'new_codon': 'Codons',
    'random_emb': 'Random', 
}
stats_df['embedding'] = stats_df['model'].map(emb_map)

stats_df

In [ ]:
color_map = {
    'significantly better': '#6C8645',
    'significantly worse': '#E3B710',
    'no significant difference': 'gray'
}

(
    ggplot(stats_df, aes(x='embedding', fill='comparison')) + 
    geom_bar(stat='count', position='dodge', color='black') +
    labs(title='nominal p-value') +
    scale_fill_manual(values=color_map) +
    theme(
        figure_size=(8,4)
        
    )
)

In [ ]:
from statsmodels.stats.multitest import multipletests

stats_df['pval_fdr'] = stats_df.groupby('model')['pval'].transform(lambda x: multipletests(x, alpha=0.05, method='fdr_by')[1])

stats_df['comparison_fdr'] = stats_df.apply(
    lambda x: 'significantly better' if (x.N_lesser < 5000) & (x.pval_fdr < 0.05) 
    else ('significantly worse' if (x.N_lesser > 5000) & (x.pval_fdr < 0.05) else 'no significant difference'), 
    axis=1
)

stats_df.to_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap_more_stats.pq', index=False)
stats_df

In [ ]:
(
    ggplot(stats_df, aes(x='embedding', fill='comparison_fdr')) + 
    geom_bar(stat='count', position='dodge', color='black') +
    scale_fill_manual(values=color_map) +
    labs(fill='Phenotype prediction\nFDR-adjusted p-value') +
    ylab('Traits') +
    xlab('') +
    theme_bw() +
    theme(
        figure_size=(10,4),
        axis_title_y=element_text(size=12),
        axis_text_x=element_text(rotation=45, hjust=1, size=11),
        legend_title=element_text(size=12),
        legend_text=element_text(size=11),
    )
)

In [ ]:
from upsetplot import from_contents, UpSet

significant_traits = {}
for embedding in stats_df.embedding.unique():
    significant_traits[embedding] = set(
        stats_df[(stats_df['embedding'] == embedding) & (stats_df['comparison_fdr'] == 'significantly better')]['trait']
    )
    
# Select three models for the Venn diagram
model1, model2, model3, model4 = 'Omics+PoPS', 'Enformer', 'gene2vec', 'ESM2'

# Prepare data for the UpSet plot
upset_data = from_contents({
    model1: significant_traits[model1],
    model2: significant_traits[model2],
    model3: significant_traits[model3],
    model4: significant_traits[model4]
})

# Create the UpSet plot
UpSet(upset_data, subset_size='count').plot()
plt.show()

# Trait-wise analysis

In [ ]:
r2_bt_multi = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap.pq')
r2_bt_multi['rel_delta_r2'] = (r2_bt_multi['r2'] - r2_bt_multi['cov_r2'])/r2_bt_multi['cov_r2']
n_bt = r2_bt_multi.bootstrap_iteration.nunique()

plot_df = r2_bt_multi[['bootstrap_iteration', 'trait', 'model', 'rel_delta_r2']].pivot(
    index=['bootstrap_iteration', 'trait'],
    columns='model',
    values='rel_delta_r2').reset_index()


# model_list = ['funcrvp', 'omics', 'pops', 'enformer', 'gene2vec', 'ESM2', 'random_emb', 'new_codon', 'new_ts2_100k', 'new_ts2_pseudobulk']
model_list = ['funcrvp', 'enformer', 'gene2vec', 'ESM2']

stats_plot_list =  []
for f_model in model_list:
    plot_df['r2_diff'] = plot_df[f_model] - plot_df['rvat']

    # --- Step 1: Grouped Aggregation ---
    # Group by the specified columns and aggregate r2_diff
    stats_plot = plot_df.groupby(['trait'])['r2_diff'].agg(
        # Count where r2_diff > 0. Boolean True is treated as 1, False as 0.
        N_greater=lambda x: (x > 0).sum(),
        # Count where r2_diff < 0
        N_lesser=lambda x: (x < 0).sum()
    ).reset_index() # reset_index turns the grouped indices back into columns

    # --- Step 2: Calculate p-value ---
    # Calculate the two terms for the minimum function
    term1 = (stats_plot['N_greater'] + 1) / (n_bt + 1)
    term2 = (stats_plot['N_lesser'] + 1) / (n_bt + 1)

    # Calculate the p-value using np.minimum for element-wise minimum
    stats_plot['pval'] = 2 * np.minimum(term1, term2)
    stats_plot['model'] = f_model
    stats_plot_list.append(stats_plot)

trait_df = pd.concat(stats_plot_list)
trait_df['comparison'] = trait_df.apply(
    lambda x: 'significantly better' if (x.N_lesser < 5000) & (x.pval < 0.05) 
    else ('significantly worse' if (x.N_lesser > 5000) & (x.pval < 0.05) else 'no significant difference'), 
    axis=1
)

trait_df['model'] = pd.Categorical(trait_df['model'], categories=model_list, ordered=True)
trait_df

In [ ]:
trait_df['embedding'] = trait_df.apply(
    lambda x: 'Omics+PoPS' if x.model == 'funcrvp' else x.model, 
    axis=1
)
trait_df

In [ ]:
from upsetplot import from_contents, UpSet

significant_traits = {}
for embedding in trait_df.embedding.unique():
    significant_traits[embedding] = set(
        trait_df[(trait_df['embedding'] == embedding) & (trait_df['comparison'] == 'significantly better')]['trait']
    )
    
# Select three models for the Venn diagram
model1, model2, model3, model4 = trait_df.embedding.unique()

# Prepare data for the UpSet plot
upset_data = from_contents({
    model1: significant_traits[model1],
    model2: significant_traits[model2],
    model3: significant_traits[model3],
    model4: significant_traits[model4]
})

# Create the UpSet plot
UpSet(upset_data, subset_size='count').plot()

plt.title('Venn Diagram of Significantly Better Traits Across Models')
plt.show()


In [ ]:
rvat_df = pd.read_parquet("/s/project/geno2pheno/funcrvp/paper_revisions/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_rvat.pq")
rvat_df['Gene\nassociations'] = rvat_df['pval'] < 0.05/rvat_df.gene_id.nunique()
rvat_ct = rvat_df.groupby('trait')['Gene\nassociations'].sum().reset_index()
rvat_ct.head()

In [ ]:
upset_data2 = upset_data.reset_index().rename(columns={'id':'trait'}).merge(rvat_ct, on='trait').set_index(['Omics+PoPS', 'enformer', 'gene2vec', 'ESM2'])
upset_data2.head()

In [ ]:
upset = UpSet(upset_data2, subset_size="count", sort_by='-cardinality')

upset.add_catplot(value="Gene\nassociations", kind="box")#, color="black")

upset.plot()
plt.title('Venn Diagram of Significantly Better Traits Across Models')
plt.show()

In [ ]:
# Calculate the number of models where comparison = 'significantly better' for each trait
significantly_better_count = trait_df[trait_df['comparison'] == 'significantly better'].groupby('trait').size().reset_index(name='num_significantly_better')

# Merge with rvat_ct to include rvat_significant
plot_data = rvat_ct.merge(significantly_better_count, on='trait', how='left').fillna(0)
plot_data['num_significantly_better'] = plot_data['num_significantly_better'].astype(str)

# Create the plot
(
    ggplot(plot_data.query("num_significantly_better != '0.0'"), aes(x='num_significantly_better', y='Gene\nassociations')) +
    geom_boxplot() +
    labs(
        title='Number of Models Significantly Better vs RVAT Significant',
        x='Number of Models Significantly Better',
        y='RVAT Significant genes'
    ) +
    theme_bw()
)